# 🔐 Hashing-Based Change Detection for RAG Indexers
## Master-Level Learning Notebook

> **Goal:** Understand and implement hashing techniques to detect when knowledge-base chunks change,
> and perform CRUD operations on your indexer so it always holds fresh, non-stale data.

---
### 📚 What you will master
| # | Topic |
|---|-------|
| 1 | Why hashing is the right primitive for change detection |
| 2 | MD5 vs SHA-256 vs xxHash — trade-offs |
| 3 | Content-addressable chunk IDs |
| 4 | Building a Hash Registry (SQLite-backed) |
| 5 | Full CRUD pipeline: detect → insert / update / delete |
| 6 | Handling chunk splits & merges (tricky edge cases) |
| 7 | Incremental sync with a simulated vector store |
| 8 | Production hardening: collision resistance, salting, versioning |
| 9 | End-to-end mini-project: live document monitor |

---

## 🧠 Section 1 — Why Hashing? The Mental Model

When your knowledge base updates, you have three choices:

```
Option A: Re-index EVERYTHING every time   → Expensive, slow, wasteful
Option B: Trust timestamps                  → Unreliable (clocks drift, files copied)
Option C: Hash every chunk                  → O(1) comparison, deterministic ✅
```

### Core insight
```
hash(chunk_text) → fixed-length fingerprint

If  hash_stored == hash_current  →  UNCHANGED  (skip)
If  hash_stored != hash_current  →  UPDATED    (re-embed & re-index)
If  hash_stored exists, chunk gone →  DELETED  (remove from index)
If  no hash_stored               →  NEW       (embed & insert)
```

This is the **content-addressable** pattern used by Git, Docker layers, and modern RAG pipelines.


In [ ]:
# ── Imports (all stdlib + commonly available) ──────────────────────────────
import hashlib
import sqlite3
import json
import uuid
import time
import copy
from datetime import datetime
from typing import Dict, List, Optional, Tuple
from dataclasses import dataclass, field, asdict
from collections import defaultdict

print('✅ All imports successful')
print(f'Python hashlib algorithms available: {sorted(hashlib.algorithms_guaranteed)}')


---
## 🔬 Section 2 — Hashing Algorithms: Choosing the Right One

| Algorithm | Digest size | Speed | Collision resistant | Use case |
|-----------|-------------|-------|--------------------|---------|
| MD5       | 128-bit     | ⚡⚡⚡  | ❌ (broken)        | Non-security checksums only |
| SHA-1     | 160-bit     | ⚡⚡   | ⚠️ (weakened)     | Avoid for new work |
| SHA-256   | 256-bit     | ⚡⚡   | ✅                  | **Recommended default** |
| SHA-512   | 512-bit     | ⚡     | ✅✅                | When paranoid |
| BLAKE2b   | 512-bit     | ⚡⚡⚡  | ✅✅                | **Best for RAG** — fast & secure |

> **RAG recommendation:** Use **BLAKE2b** truncated to 32 hex chars — fast as MD5, secure as SHA-256.


In [ ]:
# ── Algorithm comparison ───────────────────────────────────────────────────

sample_chunk = """
Retrieval-Augmented Generation (RAG) is a technique that combines a retrieval
system with a generative model. The retrieval system fetches relevant context
from a knowledge base, which the generator uses to produce accurate answers.
""".strip()

def benchmark_hash(text: str, algorithm: str, truncate: int = None) -> dict:
    encoded = text.encode('utf-8')
    start = time.perf_counter()
    iterations = 50_000
    for _ in range(iterations):
        if algorithm == 'blake2b':
            h = hashlib.blake2b(encoded, digest_size=32).hexdigest()
        else:
            h = hashlib.new(algorithm, encoded).hexdigest()
    elapsed = (time.perf_counter() - start) / iterations * 1e6
    digest = h[:truncate] if truncate else h
    return {'algorithm': algorithm, 'digest': digest, 'length': len(digest), 'us_per_call': round(elapsed, 4)}

algorithms = [
    ('md5', None), ('sha1', None), ('sha256', None), ('sha512', None), ('blake2b', 64)
]

print(f'{'Algorithm':<12} {'Digest (first 24 chars)':<28} {'Full len':>9} {'μs/call':>10}')
print('-' * 65)
for algo, trunc in algorithms:
    r = benchmark_hash(sample_chunk, algo, trunc)
    print(f"{r['algorithm']:<12} {r['digest'][:24]:<28} {r['length']:>9} {r['us_per_call']:>10}")

print('\n💡 BLAKE2b gives MD5-level speed with SHA-256-level security.')


---
## 🧩 Section 3 — Content-Addressable Chunk IDs

A **content-addressable ID** is derived from the content itself, not from position or timestamps.

```
chunk_id = hash(doc_id + chunk_index + chunk_text)
```

**Why include `doc_id` and `chunk_index`?**
- Two identical paragraphs in different docs should have different IDs (different provenance)
- Prevents false deduplication across documents

**Two common strategies:**

| Strategy | ID formula | Good for |
|----------|-----------|----------|
| Position-stable | `hash(doc_id + position)` | Fixed-length docs where content shifts |
| Content-stable  | `hash(doc_id + content)`  | Docs where content moves but doesn't change |

We'll implement both and show when each wins.


In [ ]:
# ── Chunk dataclass & ID generation ────────────────────────────────────────

@dataclass
class Chunk:
    doc_id: str
    chunk_index: int
    text: str
    metadata: dict = field(default_factory=dict)
    # Derived fields (set automatically)
    chunk_id: str = field(default='')
    content_hash: str = field(default='')

    def __post_init__(self):
        self.content_hash = self._compute_hash(self.text)
        # chunk_id = hash of (doc_id + index) — position-stable
        self.chunk_id = self._compute_hash(f'{self.doc_id}::{self.chunk_index}')

    @staticmethod
    def _compute_hash(data: str) -> str:
        return hashlib.blake2b(data.encode('utf-8'), digest_size=16).hexdigest()

    def __repr__(self):
        return (f'Chunk(doc={self.doc_id!r}, idx={self.chunk_index}, '
                f'id={self.chunk_id[:12]}…, hash={self.content_hash[:12]}…)')


# Demo
doc_chunks_v1 = [
    Chunk('doc_001', 0, 'Introduction to neural networks and deep learning basics.'),
    Chunk('doc_001', 1, 'Backpropagation is the algorithm used to train neural networks.'),
    Chunk('doc_001', 2, 'Convolutional networks excel at image recognition tasks.'),
]

print('📄 Initial document chunks:')
print(f'{"chunk_id":<34} {"content_hash":<34} {"text[:50]"}')
print('-' * 110)
for c in doc_chunks_v1:
    print(f'{c.chunk_id:<34} {c.content_hash:<34} {c.text[:50]}')


In [ ]:
# ── Demonstrate: same text → same hash (content-addressable property) ────

chunk_a = Chunk('doc_A', 0, 'Attention is all you need.')
chunk_b = Chunk('doc_B', 5, 'Attention is all you need.')  # same text, different doc

print('Same text, different documents:')
print(f'  chunk_a.content_hash = {chunk_a.content_hash}')
print(f'  chunk_b.content_hash = {chunk_b.content_hash}')
print(f'  Hashes equal? {chunk_a.content_hash == chunk_b.content_hash}  ← ✅ expected — text is identical')

print()
print('But chunk IDs differ (because doc_id differs):')
print(f'  chunk_a.chunk_id     = {chunk_a.chunk_id}')
print(f'  chunk_b.chunk_id     = {chunk_b.chunk_id}')
print(f'  IDs equal? {chunk_a.chunk_id == chunk_b.chunk_id}  ← ✅ expected — different provenance')


---
## 🗄️ Section 4 — Hash Registry: The Source of Truth

The **Hash Registry** is a persistent store that maps:
```
chunk_id  →  { content_hash, doc_id, chunk_index, last_seen, indexed_at, version }
```

Think of it as a **ledger** — every time the indexer processes a chunk, it consults and updates this ledger.

```
┌─────────────────────────────────────────────────────────────────────┐
│                         HASH REGISTRY                              │
│  chunk_id │ content_hash │ doc_id  │ idx │ last_seen │ version      │
│  ─────────┼─────────────┼─────────┼─────┼───────────┼────────────  │
│  abc123…  │  def456…    │ doc_001 │  0  │ 2024-01   │  1           │
│  bcd234…  │  efg567…    │ doc_001 │  1  │ 2024-01   │  1           │
└─────────────────────────────────────────────────────────────────────┘
```


In [ ]:
# ── Hash Registry backed by SQLite ─────────────────────────────────────────

class HashRegistry:
    """
    Persistent store for chunk content hashes.
    In production: swap SQLite for Redis, DynamoDB, or Postgres.
    """

    CREATE_SQL = """
        CREATE TABLE IF NOT EXISTS chunk_registry (
            chunk_id     TEXT PRIMARY KEY,
            content_hash TEXT NOT NULL,
            doc_id       TEXT NOT NULL,
            chunk_index  INTEGER NOT NULL,
            last_seen    TEXT NOT NULL,
            indexed_at   TEXT,
            version      INTEGER DEFAULT 1,
            metadata     TEXT DEFAULT '{}'
        );
        CREATE INDEX IF NOT EXISTS idx_doc ON chunk_registry(doc_id);
    """

    def __init__(self, db_path: str = ':memory:'):
        self.conn = sqlite3.connect(db_path)
        self.conn.row_factory = sqlite3.Row
        self.conn.executescript(self.CREATE_SQL)
        self.conn.commit()

    def get(self, chunk_id: str) -> Optional[dict]:
        row = self.conn.execute(
            'SELECT * FROM chunk_registry WHERE chunk_id = ?', (chunk_id,)
        ).fetchone()
        return dict(row) if row else None

    def upsert(self, chunk: Chunk) -> str:
        """Insert or update a chunk record. Returns 'insert' | 'update' | 'unchanged'."""
        existing = self.get(chunk.chunk_id)
        now = datetime.utcnow().isoformat()

        if existing is None:
            self.conn.execute(
                '''INSERT INTO chunk_registry
                   (chunk_id, content_hash, doc_id, chunk_index, last_seen, indexed_at, version, metadata)
                   VALUES (?, ?, ?, ?, ?, ?, 1, ?)''',
                (chunk.chunk_id, chunk.content_hash, chunk.doc_id,
                 chunk.chunk_index, now, now, json.dumps(chunk.metadata))
            )
            self.conn.commit()
            return 'insert'

        if existing['content_hash'] == chunk.content_hash:
            # Touch last_seen only
            self.conn.execute(
                'UPDATE chunk_registry SET last_seen = ? WHERE chunk_id = ?',
                (now, chunk.chunk_id)
            )
            self.conn.commit()
            return 'unchanged'

        # Hash changed → content updated
        self.conn.execute(
            '''UPDATE chunk_registry
               SET content_hash = ?, last_seen = ?, indexed_at = ?,
                   version = version + 1, metadata = ?
               WHERE chunk_id = ?''',
            (chunk.content_hash, now, now,
             json.dumps(chunk.metadata), chunk.chunk_id)
        )
        self.conn.commit()
        return 'update'

    def delete(self, chunk_id: str) -> bool:
        cur = self.conn.execute(
            'DELETE FROM chunk_registry WHERE chunk_id = ?', (chunk_id,)
        )
        self.conn.commit()
        return cur.rowcount > 0

    def get_all_chunk_ids_for_doc(self, doc_id: str) -> set:
        rows = self.conn.execute(
            'SELECT chunk_id FROM chunk_registry WHERE doc_id = ?', (doc_id,)
        ).fetchall()
        return {r['chunk_id'] for r in rows}

    def stats(self) -> dict:
        total = self.conn.execute('SELECT COUNT(*) FROM chunk_registry').fetchone()[0]
        docs  = self.conn.execute('SELECT COUNT(DISTINCT doc_id) FROM chunk_registry').fetchone()[0]
        return {'total_chunks': total, 'unique_docs': docs}

    def show_table(self, limit=20):
        rows = self.conn.execute(
            'SELECT chunk_id, content_hash, doc_id, chunk_index, version FROM chunk_registry LIMIT ?',
            (limit,)
        ).fetchall()
        print(f'{"chunk_id"[:12]:<14} {"content_hash"[:12]:<14} {"doc_id":<12} {"idx":>5} {"ver":>5}')
        print('-' * 55)
        for r in rows:
            print(f"{r['chunk_id'][:12]:<14} {r['content_hash'][:12]:<14} {r['doc_id']:<12} {r['chunk_index']:>5} {r['version']:>5}")


# Instantiate
registry = HashRegistry()  # in-memory for this notebook
print('✅ Hash Registry ready (SQLite in-memory)')
print(f'Stats: {registry.stats()}')


---
## 🔄 Section 5 — Full CRUD Pipeline

```
Knowledge Base  ──► Chunker ──► Hash Check ──► Decision ──► Indexer
                                    │
                               Hash Registry

Decision tree:
  chunk_id NOT in registry           → CREATE  (embed + insert into vector DB)
  chunk_id in registry, hash SAME    → SKIP    (no-op, just touch last_seen)
  chunk_id in registry, hash DIFF    → UPDATE  (re-embed + update vector DB)
  chunk_id in registry, NOT in source→ DELETE  (remove from vector DB + registry)
```

Let's build the full `IndexSyncer` class that orchestrates all of this.


In [ ]:
# ── Simulated Vector Store ──────────────────────────────────────────────────
# In production this would be Pinecone / Weaviate / pgvector / Chroma etc.

class SimulatedVectorStore:
    """Mimics a vector DB: stores chunk_id → {embedding, text, metadata}."""

    def __init__(self):
        self._store: Dict[str, dict] = {}
        self._ops_log: List[dict] = []

    def _fake_embed(self, text: str) -> List[float]:
        """Deterministic fake embedding (real: call OpenAI/Cohere/local model)."""
        h = hashlib.sha256(text.encode()).digest()
        return [b / 255.0 for b in h[:8]]  # 8-dim fake vector

    def insert(self, chunk: Chunk) -> None:
        self._store[chunk.chunk_id] = {
            'embedding': self._fake_embed(chunk.text),
            'text': chunk.text,
            'doc_id': chunk.doc_id,
            'chunk_index': chunk.chunk_index,
            'metadata': chunk.metadata,
        }
        self._ops_log.append({'op': 'INSERT', 'chunk_id': chunk.chunk_id[:12]})

    def update(self, chunk: Chunk) -> None:
        self._store[chunk.chunk_id] = {
            'embedding': self._fake_embed(chunk.text),
            'text': chunk.text,
            'doc_id': chunk.doc_id,
            'chunk_index': chunk.chunk_index,
            'metadata': chunk.metadata,
        }
        self._ops_log.append({'op': 'UPDATE', 'chunk_id': chunk.chunk_id[:12]})

    def delete(self, chunk_id: str) -> None:
        self._store.pop(chunk_id, None)
        self._ops_log.append({'op': 'DELETE', 'chunk_id': chunk_id[:12]})

    def count(self) -> int:
        return len(self._store)

    def print_ops(self):
        counts = defaultdict(int)
        for op in self._ops_log:
            counts[op['op']] += 1
        for op_type, n in sorted(counts.items()):
            print(f'  {op_type}: {n}')
        self._ops_log.clear()


vector_store = SimulatedVectorStore()
print('✅ Simulated vector store ready')


In [ ]:
# ── IndexSyncer: the orchestrator ───────────────────────────────────────────

class IndexSyncer:
    """
    Orchestrates change detection and CRUD between:
      - A knowledge base (list of Chunk objects)
      - A Hash Registry (change detection)
      - A Vector Store  (the actual indexer)
    """

    def __init__(self, registry: HashRegistry, vector_store: SimulatedVectorStore):
        self.registry = registry
        self.vs = vector_store

    def sync_document(self, doc_id: str, new_chunks: List[Chunk]) -> dict:
        """
        Sync a single document's chunks with the indexer.
        Returns a summary dict with counts per operation.
        """
        stats = {'insert': 0, 'update': 0, 'delete': 0, 'skip': 0}

        # ── Phase 1: figure out existing chunk IDs for this doc ─────────────
        existing_ids = self.registry.get_all_chunk_ids_for_doc(doc_id)
        incoming_ids = {c.chunk_id for c in new_chunks}

        # ── Phase 2: process incoming chunks (INSERT / UPDATE / SKIP) ───────
        for chunk in new_chunks:
            action = self.registry.upsert(chunk)
            if action == 'insert':
                self.vs.insert(chunk)
                stats['insert'] += 1
            elif action == 'update':
                self.vs.update(chunk)
                stats['update'] += 1
            else:  # unchanged
                stats['skip'] += 1

        # ── Phase 3: delete stale chunks (were indexed, no longer in source) ─
        stale_ids = existing_ids - incoming_ids
        for stale_id in stale_ids:
            self.vs.delete(stale_id)
            self.registry.delete(stale_id)
            stats['delete'] += 1

        return stats

    def sync_many(self, documents: Dict[str, List[Chunk]]) -> dict:
        """Sync multiple documents in one pass."""
        totals = {'insert': 0, 'update': 0, 'delete': 0, 'skip': 0}
        for doc_id, chunks in documents.items():
            result = self.sync_document(doc_id, chunks)
            for k in totals:
                totals[k] += result[k]
        return totals


syncer = IndexSyncer(registry, vector_store)
print('✅ IndexSyncer ready')


In [ ]:
# ── RUN 1: First sync (all new) ─────────────────────────────────────────────

print('=' * 60)
print('🔁 RUN 1: First sync — all chunks are NEW')
print('=' * 60)

doc_v1 = [
    Chunk('doc_001', 0, 'Introduction: RAG stands for Retrieval-Augmented Generation.'),
    Chunk('doc_001', 1, 'Embeddings convert text into dense numerical vectors.'),
    Chunk('doc_001', 2, 'The retriever finds semantically similar chunks.'),
    Chunk('doc_001', 3, 'The generator produces the final answer using context.'),
]

stats = syncer.sync_document('doc_001', doc_v1)
print(f'\nSync results: {stats}')
print(f'Vector store size: {vector_store.count()} chunks')
print(f'Registry stats: {registry.stats()}')
print('\nRegistry contents:')
registry.show_table()


In [ ]:
# ── RUN 2: Second sync — no changes ─────────────────────────────────────────

print('=' * 60)
print('🔁 RUN 2: Same sync again — zero changes expected')
print('=' * 60)

stats = syncer.sync_document('doc_001', doc_v1)  # Exact same content
print(f'\nSync results: {stats}')
print('\n💡 Zero re-embeddings! The hash check saved 4 API calls.')


In [ ]:
# ── RUN 3: Partial update — chunk 1 changed, chunk 3 deleted, new chunk added ─

print('=' * 60)
print('🔁 RUN 3: Mixed update (1 changed, 1 deleted, 1 new)')
print('=' * 60)

doc_v2 = [
    Chunk('doc_001', 0, 'Introduction: RAG stands for Retrieval-Augmented Generation.'),  # unchanged
    Chunk('doc_001', 1, 'Embeddings convert text into SPARSE or DENSE numerical vectors.'),  # ← CHANGED
    Chunk('doc_001', 2, 'The retriever finds semantically similar chunks.'),  # unchanged
    # chunk idx 3 is REMOVED
    Chunk('doc_001', 4, 'Re-ranking improves retrieval precision in production systems.'),  # ← NEW
]

stats = syncer.sync_document('doc_001', doc_v2)

print(f'\nSync results: {stats}')
print(f'\nExpected: insert=1, update=1, delete=1, skip=2')
assert stats['insert'] == 1
assert stats['update'] == 1
assert stats['delete'] == 1
assert stats['skip']   == 2
print('✅ All assertions pass!')
print(f'\nVector store size: {vector_store.count()} chunks')
print('\nUpdated registry:')
registry.show_table()


---
## 🧪 Section 6 — Edge Cases: Splits, Merges & Position Drift

Real-world documents are messy. Here are the three trickiest cases:

### Case A: Chunk Split
```
Before: [Chunk 0: "A. B."]       After: [Chunk 0: "A."] [Chunk 1: "B."]
```
With **position-stable IDs**: Chunk 0 looks 'updated', Chunk 1 looks 'new'. ✅ Handled.

### Case B: Content Insertion at Top (Position Drift)
```
Before: [Chunk 0: "Intro"] [Chunk 1: "Body"] [Chunk 2: "Conclusion"]
After:  [Chunk 0: "Preface"] [Chunk 1: "Intro"] [Chunk 2: "Body"] [Chunk 3: "Conclusion"]
```
With **position-stable IDs**: ALL 3 chunks look 'updated' even though content is unchanged. ❌ Bad!
With **content-stable IDs**: Only 'Preface' is new; others are detected as unchanged. ✅ Good!

### Case C: Document Deletion
A whole document disappears — all its chunks must be purged from the index.


In [ ]:
# ── Case B: Position-stable vs Content-stable ID comparison ────────────────

def content_stable_id(doc_id: str, text: str) -> str:
    """Chunk ID derived from content (not position)."""
    return hashlib.blake2b(f'{doc_id}::{text}'.encode(), digest_size=16).hexdigest()

def position_stable_id(doc_id: str, index: int) -> str:
    """Chunk ID derived from position (not content)."""
    return hashlib.blake2b(f'{doc_id}::{index}'.encode(), digest_size=16).hexdigest()


texts_v1 = ['Intro paragraph.', 'Body section.', 'Conclusion.']
texts_v2 = ['New preface!', 'Intro paragraph.', 'Body section.', 'Conclusion.']

print('=== Position-stable IDs (our Chunk class default) ===')
print(f'{"":2} {"Text":<25} {"ID v1":<14} {"ID v2":<14} {"Same?"}')
print('-' * 70)
for i, text_v2 in enumerate(texts_v2):
    id_v1 = position_stable_id('doc_X', i)[:12]
    id_v2 = position_stable_id('doc_X', i)[:12]
    # The IDs are same (same position) but content may differ
    text_v1 = texts_v1[i] if i < len(texts_v1) else '(new)'
    same_content = text_v1 == text_v2
    flag = '✅' if same_content else '⚠️  WOULD RE-EMBED'
    print(f'{i:<2} v1={text_v1[:18]:<22} v2={text_v2[:18]:<22} {flag}')

print()
print('=== Content-stable IDs ===')
print(f'{"":2} {"Text":<25} {"ID v1":<14} {"ID v2":<14} {"Recognized?"}')
print('-' * 75)

ids_v1 = {content_stable_id('doc_X', t)[:12]: t for t in texts_v1}
for text in texts_v2:
    cid = content_stable_id('doc_X', text)[:12]
    recognized = cid in ids_v1
    flag = '✅ SKIP (unchanged)' if recognized else '🆕 NEW'
    print(f'   {text:<28} id={cid}  {flag}')

print('\n💡 Content-stable IDs save 3 re-embeddings when a preface is prepended!')


In [ ]:
# ── Case C: Full document deletion ──────────────────────────────────────────

class IndexSyncerV2(IndexSyncer):
    """Extended syncer with document deletion support."""

    def delete_document(self, doc_id: str) -> int:
        """Remove all chunks of a document from vector store and registry."""
        chunk_ids = self.registry.get_all_chunk_ids_for_doc(doc_id)
        for cid in chunk_ids:
            self.vs.delete(cid)
            self.registry.delete(cid)
        return len(chunk_ids)


# Set up fresh state
reg2 = HashRegistry()
vs2  = SimulatedVectorStore()
syncer2 = IndexSyncerV2(reg2, vs2)

# Index two documents
syncer2.sync_document('annual_report', [
    Chunk('annual_report', 0, 'Revenue grew 22% YoY in fiscal 2024.'),
    Chunk('annual_report', 1, 'Operating margin improved to 31%.'),
])
syncer2.sync_document('product_spec', [
    Chunk('product_spec', 0, 'The widget supports USB-C and Bluetooth 5.2.'),
    Chunk('product_spec', 1, 'Battery life is rated at 48 hours.'),
])

print(f'Before deletion → registry: {reg2.stats()}, vector store: {vs2.count()} chunks')

# Now the annual report is retracted
deleted = syncer2.delete_document('annual_report')

print(f'After deletion  → registry: {reg2.stats()}, vector store: {vs2.count()} chunks')
print(f'Deleted {deleted} chunks from annual_report.')
print('✅ Product spec chunks remain unaffected.')


---
## 🔁 Section 7 — Incremental Sync: The Full Pipeline

Real RAG systems often need to:
1. **Crawl** a knowledge base (files, URLs, DB rows)
2. **Chunk** each document
3. **Hash** and diff against the registry
4. **Execute** only the necessary CRUD operations

Below is a complete, runnable simulation of this pipeline with metrics.


In [ ]:
# ── Simulation: evolving knowledge base over 4 sync rounds ────────────────

import random
random.seed(42)

# Simulate a knowledge base that changes over time
KB_ROUNDS = {
    'round_1': {
        'faq': [
            'Q: What is the refund policy? A: 30-day no-questions-asked refund.',
            'Q: Do you ship internationally? A: Yes, to 45 countries.',
            'Q: What payment methods do you accept? A: Visa, Mastercard, PayPal.',
        ],
        'product': [
            'Product A: Industrial-grade wireless sensor, rated IP67.',
            'Product B: Solar-powered data logger with 4G connectivity.',
        ],
    },
    'round_2': {  # FAQ updated, new doc added
        'faq': [
            'Q: What is the refund policy? A: 30-day no-questions-asked refund.',  # same
            'Q: Do you ship internationally? A: Yes, to 52 countries.',  # UPDATED
            'Q: What payment methods do you accept? A: Visa, Mastercard, PayPal.',  # same
            'Q: Is there a loyalty programme? A: Yes! Earn 1 point per dollar.',  # NEW
        ],
        'product': [
            'Product A: Industrial-grade wireless sensor, rated IP67.',  # same
            'Product B: Solar-powered data logger with 4G and 5G connectivity.',  # UPDATED
        ],
        'pricing': [
            'Starter plan: $49/month, up to 10 devices.',  # NEW doc
            'Growth plan: $149/month, up to 50 devices.',
        ],
    },
    'round_3': {  # No changes at all
        'faq': [
            'Q: What is the refund policy? A: 30-day no-questions-asked refund.',
            'Q: Do you ship internationally? A: Yes, to 52 countries.',
            'Q: What payment methods do you accept? A: Visa, Mastercard, PayPal.',
            'Q: Is there a loyalty programme? A: Yes! Earn 1 point per dollar.',
        ],
        'product': [
            'Product A: Industrial-grade wireless sensor, rated IP67.',
            'Product B: Solar-powered data logger with 4G and 5G connectivity.',
        ],
        'pricing': [
            'Starter plan: $49/month, up to 10 devices.',
            'Growth plan: $149/month, up to 50 devices.',
        ],
    },
    'round_4': {  # pricing doc removed, faq trimmed
        'faq': [
            'Q: What is the refund policy? A: 30-day no-questions-asked refund.',
            'Q: Do you ship internationally? A: Yes, to 52 countries.',
            # payment method question REMOVED
            'Q: Is there a loyalty programme? A: Yes! Earn 1 point per dollar.',
        ],
        'product': [
            'Product A: Industrial-grade wireless sensor, rated IP67.',
            'Product B: Solar-powered data logger with 4G and 5G connectivity.',
        ],
        # pricing doc DELETED entirely
    },
}

reg3 = HashRegistry()
vs3  = SimulatedVectorStore()
syncer3 = IndexSyncerV2(reg3, vs3)

# Track which docs existed last round (for deletion detection)
prev_doc_ids = set()

print(f'{'Round':<8} {'Insert':>8} {'Update':>8} {'Delete':>8} {'Skip':>8} {'VectorDB':>10} {'API calls saved':>16}')
print('-' * 75)

for round_name, kb in KB_ROUNDS.items():
    current_doc_ids = set(kb.keys())

    # Detect deleted documents
    for doc_id in prev_doc_ids - current_doc_ids:
        syncer3.delete_document(doc_id)

    # Build chunks for this round and sync
    round_docs = {}
    for doc_id, texts in kb.items():
        round_docs[doc_id] = [Chunk(doc_id, i, t) for i, t in enumerate(texts)]

    stats = syncer3.sync_many(round_docs)
    total_chunks = sum(len(v) for v in round_docs.values())
    ops_needed   = stats['insert'] + stats['update']
    saved        = total_chunks - ops_needed

    print(f"{round_name:<8} {stats['insert']:>8} {stats['update']:>8} {stats['delete']:>8} {stats['skip']:>8} {vs3.count():>10} {saved:>14} ({saved}/{total_chunks})")
    prev_doc_ids = current_doc_ids

print(f'\n✅ Final registry: {reg3.stats()}')


---
## 🛡️ Section 8 — Production Hardening

### 8.1 Salted Hashes (prevent pre-image attacks on sensitive docs)
```python
content_hash = blake2b(salt + doc_id + text)
```

### 8.2 Collision Probability
BLAKE2b-128 (16 bytes): probability of any collision across 1 billion chunks ≈ 2^-64 ≈ **negligible**.

### 8.3 Versioning & Rollback
Keep previous hash in registry → enables one-step rollback.

### 8.4 Chunking Strategy Matters
If you change your chunking strategy (e.g., size 512→1024 tokens), ALL hashes change. 
**Include chunk strategy version in the hash input.**

### 8.5 Metadata Changes
Sometimes metadata (page number, source URL) changes but text doesn't. 
Decide: does metadata change require re-embedding? Usually NO — just update the doc store.


In [ ]:
# ── 8.1  Salted hashing ─────────────────────────────────────────────────────

SALT = b'my-rag-pipeline-v1'  # Store this securely; change = all hashes invalidated

def salted_hash(text: str, doc_id: str, salt: bytes = SALT) -> str:
    payload = salt + doc_id.encode() + text.encode()
    return hashlib.blake2b(payload, digest_size=16).hexdigest()

h1 = salted_hash('Price is $100', 'catalog')
h2 = salted_hash('Price is $100', 'catalog')
h3 = salted_hash('Price is $100', 'catalog', salt=b'different-salt')
print(f'Same inputs, same salt: {h1 == h2}  ← ✅')
print(f'Same inputs, diff salt: {h1 == h3}  ← ✅ (salt isolates environments)')
print(f'h1: {h1}')
print(f'h3: {h3}')


In [ ]:
# ── 8.4  Chunk-strategy version baked into hash ──────────────────────────────

def versioned_hash(text: str, doc_id: str, chunk_strategy_version: str = 'v1') -> str:
    """
    If you change chunking strategy, bump version → all old hashes mismatch
    → full re-index triggered automatically.
    """
    payload = f'{chunk_strategy_version}::{doc_id}::{text}'.encode()
    return hashlib.blake2b(payload, digest_size=16).hexdigest()

h_v1 = versioned_hash('Some text chunk', 'doc_001', 'v1')
h_v2 = versioned_hash('Some text chunk', 'doc_001', 'v2')  # strategy changed

print(f'v1 hash: {h_v1}')
print(f'v2 hash: {h_v2}')
print(f'Different? {h_v1 != h_v2}  ← ✅ forces re-index when strategy changes')


In [ ]:
# ── 8.5  Metadata-only changes ──────────────────────────────────────────────

@dataclass
class ChunkV2:
    doc_id: str
    chunk_index: int
    text: str
    metadata: dict = field(default_factory=dict)
    chunk_id: str = field(default='')
    content_hash: str = field(default='')   # text only
    full_hash: str = field(default='')      # text + metadata

    def __post_init__(self):
        self.content_hash = hashlib.blake2b(
            self.text.encode(), digest_size=16
        ).hexdigest()
        meta_str = json.dumps(self.metadata, sort_keys=True)
        self.full_hash = hashlib.blake2b(
            (self.text + meta_str).encode(), digest_size=16
        ).hexdigest()
        self.chunk_id = hashlib.blake2b(
            f'{self.doc_id}::{self.chunk_index}'.encode(), digest_size=16
        ).hexdigest()

c_old = ChunkV2('doc_X', 0, 'Transformers use self-attention.', {'page': 1})
c_new = ChunkV2('doc_X', 0, 'Transformers use self-attention.', {'page': 2})  # page changed

print('Content hash same?', c_old.content_hash == c_new.content_hash, '← text unchanged, NO re-embed needed')
print('Full hash same?   ', c_old.full_hash    == c_new.full_hash,    '← includes metadata, trigger metadata update')
print()
print('💡 Strategy:')
print('  content_hash differs → re-embed + re-index (expensive)')
print('  only full_hash differs → update metadata in doc store (cheap)')


---
## 🏗️ Section 9 — End-to-End Mini Project: Document Monitor

We'll simulate a **file-based knowledge base** where documents change over time,
and our monitor continuously syncs the index.

Architecture:
```
files/ ──► DocumentCrawler ──► Chunker ──► IndexSyncer
                                               │
                                         HashRegistry ── SQLite
                                               │
                                         VectorStore
```


In [ ]:
# ── Document Monitor: full end-to-end system ────────────────────────────────

class DocumentCrawler:
    """Simulates reading files from disk (in-memory for demo)."""

    def __init__(self):
        self._files: Dict[str, str] = {}

    def write(self, path: str, content: str):
        self._files[path] = content

    def delete(self, path: str):
        self._files.pop(path, None)

    def crawl(self) -> Dict[str, str]:
        return dict(self._files)


class NaiveChunker:
    """Split text on double-newlines."""

    def chunk(self, doc_id: str, text: str) -> List[Chunk]:
        paragraphs = [p.strip() for p in text.split('\n\n') if p.strip()]
        return [Chunk(doc_id, i, p) for i, p in enumerate(paragraphs)]


class DocumentMonitor:
    """Orchestrates the full pipeline."""

    def __init__(self):
        self.crawler = DocumentCrawler()
        self.chunker = NaiveChunker()
        self.registry = HashRegistry()
        self.vector_store = SimulatedVectorStore()
        self.syncer = IndexSyncerV2(self.registry, self.vector_store)
        self._prev_paths: set = set()
        self.sync_count = 0

    def run_sync(self):
        self.sync_count += 1
        print(f'\n{'─'*55}')
        print(f'⏱  Sync #{self.sync_count} at {datetime.utcnow().strftime("%H:%M:%S")} UTC')
        print(f'{'─'*55}')

        files = self.crawler.crawl()
        current_paths = set(files.keys())

        # Delete removed docs
        for path in self._prev_paths - current_paths:
            n = self.syncer.delete_document(path)
            print(f'  🗑  Deleted document {path!r} ({n} chunks removed)')

        # Build & sync
        docs = {path: self.chunker.chunk(path, text) for path, text in files.items()}
        stats = self.syncer.sync_many(docs)

        print(f'  📊 INSERT={stats["insert"]}  UPDATE={stats["update"]}  '
              f'DELETE={stats["delete"]}  SKIP={stats["skip"]}')
        print(f'  📦 Vector store: {self.vector_store.count()} chunks | '
              f'Registry: {self.registry.stats()["total_chunks"]} entries')

        self._prev_paths = current_paths
        return stats


monitor = DocumentMonitor()
print('✅ DocumentMonitor ready')


In [ ]:
# ── Simulate 5 sync events ──────────────────────────────────────────────────

# Event 1: Initial state
monitor.crawler.write('docs/intro.txt', """
Welcome to AcmeCorp's knowledge base.

This portal contains FAQs, product specs, and support guides.

Last updated by the documentation team.
""".strip())

monitor.crawler.write('docs/products.txt', """
Product Alpha: Industrial IoT gateway, supports MQTT and REST.

Product Beta: Edge AI inference module, 8 TOPS performance.
""".strip())

monitor.run_sync()  # Sync 1 — all new

# Event 2: No changes
monitor.run_sync()  # Sync 2 — all skip

# Event 3: Products doc updated
monitor.crawler.write('docs/products.txt', """
Product Alpha: Industrial IoT gateway, supports MQTT and REST.

Product Beta: Edge AI inference module, 12 TOPS performance.  ← upgraded!

Product Gamma: Cloud-to-edge orchestration platform.  ← NEW PRODUCT
""".strip())
monitor.run_sync()  # Sync 3 — 1 update, 1 insert

# Event 4: New support doc added
monitor.crawler.write('docs/support.txt', """
For technical support, email support@acmecorp.com.

Phone support available Mon-Fri 9am-6pm EST.
""".strip())
monitor.run_sync()  # Sync 4 — 2 new chunks

# Event 5: intro.txt deleted, products.txt untouched
monitor.crawler.delete('docs/intro.txt')
monitor.run_sync()  # Sync 5 — 3 deletions

print(f'\n{'='*55}')
print(f'✅ Simulation complete.')
print(f'   Final registry: {monitor.registry.stats()}')
print(f'   Final vector store: {monitor.vector_store.count()} chunks')


---
## 🎓 Summary & Production Cheatsheet

### Core principles you've mastered

| Principle | Implementation |
|-----------|---------------|
| Hash every chunk | `BLAKE2b(doc_id + text, digest_size=16)` |
| Separate chunk_id from content_hash | `chunk_id = hash(doc_id + position)` |
| Only act on changes | Compare stored hash vs current hash |
| Version your strategy | Include `chunk_strategy_version` in hash |
| Salt for security | Include environment salt in hash |
| Separate text vs metadata changes | Two hash fields: `content_hash`, `full_hash` |
| Track doc lifecycle | Maintain `prev_doc_ids` to detect deletions |

### Decision flowchart

```
For each chunk in source:
  ├─ chunk_id NOT in registry?       → EMBED + INSERT into vector DB + registry
  ├─ content_hash unchanged?         → touch last_seen only (SKIP)
  └─ content_hash changed?           → RE-EMBED + UPDATE vector DB + bump version

For each chunk_id in registry (for this doc):
  └─ NOT found in current source?    → DELETE from vector DB + registry
```

### Production stack recommendation

```
Hash store:    Redis (sub-ms lookups) or Postgres (JSONB column for metadata)
Vector store:  Pinecone / Weaviate / pgvector / Chroma
Hash algo:     BLAKE2b-128 (fast + collision-resistant)
Scheduler:     Cron / Airflow / Celery Beat for periodic sync
Observability: Track insert/update/delete/skip rates over time
```

### Anti-patterns to avoid

```
❌ Using MD5 or SHA-1 (broken collision resistance)
❌ Using file mtime (unreliable — clocks drift, files get copied)
❌ Hashing after whitespace normalisation only (miss subtle edits)
❌ Same chunk_id formula for different environments (use salt!)
❌ Not handling document-level deletions (stale chunks accumulate)
❌ Changing chunking strategy without bumping the version (invisible full re-index)
```

---
*Notebook built for mastering hashing-based RAG index synchronisation — © 2024*
